In [0]:
%pip install xgboost scikit-learn hyperopt

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.models")

In [0]:
import json
import mlflow
import xgboost as xgb
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/marcus.egelund-muller@devoteam.com/xgb_combined_direction")

In [0]:
df = (
    spark.table("stocks.gold.stocks_combined_features")
    .dropna()          # drop rows where FX/macro not yet available
    .orderBy("Date")   # ensure chronological order before split
    .toPandas()
)

EXCLUDE = ["label", "Date", "company"]
feature_cols = [c for c in df.columns if c not in EXCLUDE]

X = df[feature_cols]
y = df["label"]

# Chronological 80/20 split — never shuffle time series
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Training rows: {len(X_train)}  |  Test rows: {len(X_test)}")
print(f"Features: {len(feature_cols)}")

In [0]:
search_space = {
    "max_depth":        hp.choice("max_depth", [3, 5, 7]),
    "learning_rate":    hp.loguniform("learning_rate", -3, -1),
    "subsample":        hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
    "n_estimators":     hp.choice("n_estimators", [200, 300, 400]),
}

def objective(params):
    with mlflow.start_run(nested=True):
        model = xgb.XGBClassifier(
            **params,
            eval_metric="logloss",
            early_stopping_rounds=20,
            tree_method="hist",
            n_jobs=-1,
        )
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        mlflow.log_metrics({"roc_auc": auc})
        return {"loss": -auc, "status": STATUS_OK, "model": model}

trials = Trials()
with mlflow.start_run(run_name="hyperopt_search_combined"):
    fmin(fn=objective, space=search_space, algo=tpe.suggest,
         max_evals=20, trials=trials)

best_idx = trials.losses().index(min(trials.losses()))
model    = trials.results[best_idx]["model"]
best_auc = -min(trials.losses())
print(f"Best ROC-AUC: {best_auc:.4f}")

# Register best model in Unity Catalog
with mlflow.start_run(run_name="combined_model_final") as run:
    acc = accuracy_score(y_test, model.predict(X_test))
    mlflow.log_metrics({"accuracy": acc, "roc_auc": best_auc})
    mlflow.log_param("feature_cols", json.dumps(feature_cols))
    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name="stocks.models.combined_predictor",
        input_example=X_train.head(1),
    )
    print(f"Run ID   : {run.info.run_id}")
    print(f"Accuracy : {acc:.4f}")
    print(f"ROC-AUC  : {best_auc:.4f}")
print("Next: set the 'champion' alias in Databricks > Models")

In [0]:
# Top-20 feature importances — verify FX and macro features appear
import pandas as pd
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.nlargest(20).sort_values().plot.barh(figsize=(8, 8), title="Top 20 Feature Importances")
display()

In [0]:
# After running this notebook, go to Databricks > Models > stocks.models.combined_predictor
# and set the 'champion' alias on the newly registered version before running predict_combined.